In [ ]:
!pip install git+https://github.com/NVlabs/tiny-cuda-nn/#subdirectory=bindings/torch
!pip install commentjson
!pip install ninja imageio PyOpenGL glfw xatlas gdown
!pip install git+https://github.com/NVlabs/nvdiffrast/
!imageio_download_bin freeimage
!pip install gpytoolbox opencv-python trimesh matplotlib chumpy lpips tqdm
!pip install git+https://github.com/jonbarron/robust_loss_pytorch
!pip install 'git+https://github.com/facebookresearch/pytorch3d.git'
!pip install open3d
!pip install configargparse

In [ ]:
!git clone https://github.com/HannesLeonhard/flare_adl4cv.git
%cd flare_adl4cv

In [ ]:
import importlib
from arguments import config_parser
import os
from pathlib import Path
import torch
from flame.FLAME import FLAME
from flare.core import (
    Mesh, Renderer
)
from flare.modules import (
    NeuralShader, get_deformer_network
)
from flare.utils import (
    AABB, read_mesh,
    save_individual_img, make_dirs, save_relit_intrinsic_materials
)
import nvdiffrec.render.light as light
from flare.dataset import DatasetLoader
from flare.dataset import dataset_util
from flare.metrics import metrics

from material_aware_flare.eval import extract_flare_channels
from material_aware_flare.utils import save_extracted_channels

# Select the device
device = torch.device('cpu')
devices = 0
if torch.cuda.is_available() and devices >= 0:
    device = torch.device(f'cuda:{devices}')

flame_path = "/content/flare_adl4cv/flame/FLAME2020/generic_model.pkl"

In [ ]:
from pathlib import Path

config = {
    'config': None,
    # run_name
    'run_name': '001',
    # batch_size
    'batch_size': 2,
    # path
    'input_dir': Path("/content/flare_dataset/DATA/001"),
    'train_dir': ["MVI_1810", "MVI_1811", "MVI_1812"],
    'eval_dir': ["MVI_1814"],
    'working_dir': Path("./"),
    'output_dir': Path("/content/flare_adl4cv_models/MODELS/"),
    # misc
    'sample_idx_ratio': 1,
    'device': 0,
    'finetune_color': False,
    # iters
    'iterations': 2000,
    'final_iter': 1500,
    'upsample_iterations': [500],
    'save_frequency': 300,
    'visualization_frequency': 100,
    'visualization_views': [15, 25, 27, 21, 26],
    # 'downsample' default is set via set_defaults
    'downsample': False,
    'downsample_ratio': 0.03,
    'grad_scale': False, # Default for action='store_true' is False unless set otherwise
    # flame
    'decay_flame': [100],
    'flame_mask': False,
    # lr
    'lr_vertices': 1e-3,
    'lr_shader': 1e-3,
    'lr_deformer': 1e-3,
    # loss weights
    'weight_mask': 2.0,
    'weight_normal': 0.1,
    'weight_laplacian': 60.0,
    'weight_shading': 1.0,
    'weight_perceptual_loss': 0.1,
    'weight_albedo_regularization': 0.01,
    'weight_flame_regularization': 10.0,
    'weight_white_lgt_regularization': 1.0,
    'weight_roughness_regularization': 0.1,
    'weight_fresnel_coeff': 0.01,
    'r_mean': 0.500,
    # neural shader
    'fourier_features': 'positional',
    'activation': 'relu',
    'bsdf': 'pbr_shading',
    'deform_d_out': 128,
    'light_mlp_ch': 3,
    'light_mlp_dims': [64, 64],
    'material_mlp_dims': [128, 128, 128, 128],
    'material_mlp_ch': 4,
    # ghostbone/train_deformer (defaults are set via set_defaults)
    'ghostbone': True,
    'train_deformer': True,
    'deform_dims': [128, 128, 128, 128]
}


subject = 1

new_values_1 = {
    'run_name': '001',
    'input_dir': Path("/content/flare_dataset/DATA/001"),
    'train_dir': ["MVI_1810", "MVI_1811", "MVI_1812"],
    'eval_dir': ["MVI_1814"],
    'working_dir': Path("/content/flare_adl4cv_models/MODELS/"),
    'output_dir': Path("/content/flare_adl4cv_models/MODELS/001"),
    'batch_size': 1,
    'sample_idx_ratio': 1,
    'iterations': 1500,
    'upsample_iterations': [500],
    'lr_deformer': 1e-3,
    'lr_shader': 1e-3,
    'lr_vertices': 1e-3,
    'weight_shading': 1.0,
    'weight_perceptual_loss': 0.1,
    'weight_mask': 2.0,
    'weight_albedo_regularization': 0.01,
    'weight_white_lgt_regularization': 0.01,
    'weight_roughness_regularization': 0.01,
    'weight_fresnel_coeff': 0.01,
    'weight_normal': 0.1,
    'weight_laplacian': 60.0,
    'light_mlp_ch': 3,
    'light_mlp_dims': [64, 64],
    'material_mlp_dims': [128, 128, 128, 128],
    'material_mlp_ch': 5
}

if subject == 1:
  config.update(new_values_1)

from types import SimpleNamespace
config = SimpleNamespace(**config)

In [ ]:
images_save_path, images_eval_save_path, meshes_save_path, shaders_save_path, experiment_dir = make_dirs(
    config, config.run_name, config.finetune_color
  )
dataset_val = DatasetLoader(config, train_dir=config.eval_dir, sample_ratio=config.sample_idx_ratio, pre_load=False)
dataloader_validate = torch.utils.data.DataLoader(dataset_val, batch_size=4, collate_fn=dataset_val.collate, shuffle=False)

In [ ]:
flame_shape = dataset_val.shape_params
FLAMEServer = FLAME(flame_path, n_shape=100, n_exp=50, shape_params=flame_shape).to(device)

In [ ]:
### init flame and deformation
flame_shape = dataset_val.shape_params
FLAMEServer = FLAME(flame_path, n_shape=100, n_exp=50, shape_params=flame_shape).to(device)
### Obtain the initial mesh and compute its connectivity
flame_canonical_mesh = Mesh(FLAMEServer.v_template, FLAMEServer.faces_tensor, device=device)
flame_canonical_mesh.compute_connectivity()
### create bounding box from the mesh vertices
aabb = AABB(flame_canonical_mesh.vertices.cpu().numpy())
flame_mesh_aabb = [torch.min(flame_canonical_mesh.vertices, dim=0).values, torch.max(flame_canonical_mesh.vertices, dim=0).values]
# init mesh is mouth open!!!
FLAMEServer.canonical_exp = dataset_val.get_mean_expression_train(config.train_dir).to(device)
FLAMEServer.canonical_pose = FLAMEServer.canonical_pose.to(device)
FLAMEServer.canonical_verts, FLAMEServer.canonical_pose_feature, FLAMEServer.canonical_transformations = \
    FLAMEServer(expression_params=FLAMEServer.canonical_exp, full_pose=FLAMEServer.canonical_pose)
FLAMEServer.canonical_verts = FLAMEServer.canonical_verts.to(device)
flame_canonical_mesh.vertices = FLAMEServer.canonical_verts.squeeze(0)

In [ ]:
# ==============================================================================================
# mesh
# ==============================================================================================
experiment_dir = Path("/content/flare_adl4cv_models/MODELS/001")
mesh_path = Path(experiment_dir / "stage_1" / "meshes" / f"mesh_latest.obj")
mesh = read_mesh(mesh_path, device=device)
mesh.compute_connectivity()
mesh.to(device)

In [ ]:
# ==============================================================================================
# Rendererrr
# ==============================================================================================
renderer = Renderer(device=device)
renderer.set_near_far(dataset_val, torch.from_numpy(aabb.corners).to(device), epsilon=0.5)
channels_gbuffer = ['mask', 'position', 'normal', "canonical_position"]

In [ ]:
# ==============================================================================================
# deformation
# ==============================================================================================
load_deformer = Path(experiment_dir / "stage_2" / "network_weights" / f"deformer_latest.pt")
assert os.path.exists(load_deformer)
multires = 0
deformer_net = get_deformer_network(FLAMEServer, model_path=load_deformer, train=False, d_in=3, dims=[128, 128, 128, 128],
                                       weight_norm=True, multires=multires, num_exp=50, aabb=aabb, ghostbone=config.ghostbone, device=device)
if config.ghostbone:
    FLAMEServer.canonical_transformations = torch.cat([torch.eye(4).unsqueeze(0).unsqueeze(0).float().to(device), FLAMEServer.canonical_transformations], 1)

In [ ]:
# ==============================================================================================
# shading
# ==============================================================================================
load_shader = Path(experiment_dir / "stage_2" / "network_weights" / f"shader_latest.pt")
assert os.path.exists(load_shader)
shader = NeuralShader.load(load_shader, device=device)
lgt = light.create_env_rnd()
print("=="*50)
shader.eval()
deformer_net.eval()
batch_size = config.batch_size
print("Batch Size:", batch_size)

In [ ]:
# ==============================================================================================
# evaluation
# ==============================================================================================
def run(
    args,
    mesh,
    views,
    FLAMEServer,
    deformer_net,
    shader,
    renderer,
    device,
    channels_gbuffer,
    lgt,
):
    ## ============== deform ==============================
    shapedirs, posedirs, lbs_weights = deformer_net.query_weights(mesh.vertices)
    eval_vertices = mesh.vertices
    batched_verts = eval_vertices.unsqueeze(0).repeat(views["img"].shape[0], 1, 1)

    _, pose_features, transformations = FLAMEServer(
        expression_params=views["flame_expression"], full_pose=views["flame_pose"]
    )
    if args.ghostbone:
        transformations = torch.cat(
            [
                torch.eye(4)
                .unsqueeze(0)
                .unsqueeze(0)
                .expand(views["img"].shape[0], -1, -1, -1)
                .float()
                .to(device),
                transformations,
            ],
            1,
        )
    deformed_vertices = FLAMEServer.forward_pts_batch(
        pnts_c=batched_verts,
        betas=views["flame_expression"],
        transformations=transformations,
        pose_feature=pose_features,
        shapedirs=shapedirs,
        posedirs=posedirs,
        lbs_weights=lbs_weights,
        dtype=torch.float32,
        map2_flame_original=True,
    )

    d_normals = mesh.fetch_all_normals(deformed_vertices, mesh)
    ## ============== Rasterize ==============================
    gbuffers = renderer.render_batch(
        views["camera"],
        deformed_vertices.contiguous(),
        d_normals,
        channels=channels_gbuffer,
        with_antialiasing=True,
        canonical_v=mesh.vertices,
        canonical_idx=mesh.indices,
    )

    ## ============== predict color ==============================
    rgb_pred, cbuffers, gbuffer_mask = shader.shade(
        gbuffers, views, mesh, args.finetune_color, lgt
    )

    return rgb_pred, gbuffers, cbuffers


# ==============================================================================================
# relight: run
# ==============================================================================================
def run_relight(
    args,
    mesh,
    views,
    FLAMEServer,
    deformer_net,
    shader,
    renderer,
    device,
    channels_gbuffer,
    lgt_list,
    images_save_path,
):
    ## ============== deform ==============================
    shapedirs, posedirs, lbs_weights = deformer_net.query_weights(mesh.vertices)
    eval_vertices = mesh.vertices
    batched_verts = eval_vertices.unsqueeze(0).repeat(views["img"].shape[0], 1, 1)

    _, pose_features, transformations = FLAMEServer(
        expression_params=views["flame_expression"], full_pose=views["flame_pose"]
    )
    if args.ghostbone:
        transformations = torch.cat(
            [
                torch.eye(4)
                .unsqueeze(0)
                .unsqueeze(0)
                .expand(views["img"].shape[0], -1, -1, -1)
                .float()
                .to(device),
                transformations,
            ],
            1,
        )
    deformed_vertices = FLAMEServer.forward_pts_batch(
        pnts_c=batched_verts,
        betas=views["flame_expression"],
        transformations=transformations,
        pose_feature=pose_features,
        shapedirs=shapedirs,
        posedirs=posedirs,
        lbs_weights=lbs_weights,
        dtype=torch.float32,
        map2_flame_original=True,
    )

    d_normals = mesh.fetch_all_normals(deformed_vertices, mesh)
    ## ============== Rasterize ==============================
    gbuffers = renderer.render_batch(
        views["camera"],
        deformed_vertices.contiguous(),
        d_normals,
        channels=channels_gbuffer,
        with_antialiasing=True,
        canonical_v=mesh.vertices,
        canonical_idx=mesh.indices,
    )

    ## ============== predict color ==============================
    relit_imgs, cbuffers, gbuffer_mask = shader.relight(
        gbuffers, views, mesh, args.finetune_color, lgt_list
    )
    save_relit_intrinsic_materials(
        relit_imgs, views, gbuffer_mask, cbuffers, images_save_path
    )

In [ ]:
# ==============================================================================================
# evaluation: intrinsic materials and relighting
# ==============================================================================================
lgt_list = light.load_target_cubemaps(config.working_dir)
for i in range(len(lgt_list)):
    Path(images_eval_save_path / "qualitative_results" / f"env_map_{i}" ).mkdir(parents=True, exist_ok=True)
for it, views_subset in enumerate(dataloader_validate):
    with torch.no_grad():
        run_relight(config, mesh, views_subset, FLAMEServer, deformer_net, shader, renderer, device, channels_gbuffer, lgt_list, images_eval_save_path / "qualitative_results")

In [ ]:
def quantitative_eval_flare(
    args,
    mesh,
    dataloader_validate,
    FLAMEServer,
    deformer_net,
    shader,
    renderer,
    device,
    channels_gbuffer,
    experiment_dir,
    images_eval_save_path,
    lgt=None,
    save_each=False,
):

    for it, views_subset in enumerate(dataloader_validate):
        with torch.no_grad():
            rgb_pred, gbuffer, cbuffer = run(
                args,
                mesh,
                views_subset,
                FLAMEServer,
                deformer_net,
                shader,
                renderer,
                device,
                channels_gbuffer,
                lgt=lgt,
            )
            extracted_channels = extract_flare_channels(
                shader, gbuffer, views_subset, mesh
            )
        save_extracted_channels(
            extracted_channels, views_subset["img_idx"], images_eval_save_path
        )
        rgb_pred = rgb_pred * gbuffer["mask"]